# 9장. 공개 모델 읽기

Hugging Face에 공개된 작은 언어 모델 네 개의 설정, 라이선스, 토크나이저, 채팅 템플릿을 읽고, 세 모델에 같은 질문을 던져 답을 비교한다.

**준비:** `pip install "transformers<5" tokenizers huggingface_hub`. 모델은 아래 첫 셀이 `data/models/`에 내려받는다(세 모델 가중치 약 7.3GB). EXAONE은 라이선스가 연구·교육 목적만 허용하므로 이 장에서는 설정·모델 카드·라이선스만 받는다. NSMC의 `ratings_train.txt`, `ratings_test.txt`도 필요하다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 9분쯤 걸렸다(내려받는 시간 제외).

## 9.1 모델 카드와 설정 읽기

In [ ]:
import os, re, json, math, time, sys, platform, warnings
import numpy as np
import torch
os.environ["RAYON_NUM_THREADS"] = "2"
os.environ["HF_HUB_DISABLE_XET"] = "1"
warnings.filterwarnings("ignore")
torch.set_num_threads(2)
import transformers, tokenizers, huggingface_hub
from huggingface_hub import snapshot_download, HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM

ROOT = "data/models"
MODELS = {"Qwen3-0.6B": "Qwen/Qwen3-0.6B",
          "HyperCLOVAX-SEED-0.5B": "naver-hyperclovax/HyperCLOVAX-SEED-Text-Instruct-0.5B",
          "EXAONE-4.0-1.2B": "LGAI-EXAONE/EXAONE-4.0-1.2B",
          "Kanana-1.5-2.1B": "kakaocorp/kanana-1.5-2.1b-instruct-2505"}
RUN = ["Qwen3-0.6B", "HyperCLOVAX-SEED-0.5B", "Kanana-1.5-2.1B"]   # 가중치까지 받아 돌리는 모델
path = {}
for name, repo in MODELS.items():
    if name in RUN:
        files = ["*.json", "*.txt", "*.md", "LICENSE*", "*.jinja", "*.safetensors"]
    else:                                                     # EXAONE은 라이선스가 연구·교육 목적만 허용하므로 설정, 모델 카드, 라이선스만 받는다
        files = ["config.json", "README.md", "LICENSE"]
    path[name] = snapshot_download(repo, local_dir=os.path.join(ROOT, repo.replace("/", "__")), allow_patterns=files)
print(sys.version.split()[0], torch.__version__, transformers.__version__, tokenizers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
conf = {n: json.load(open(os.path.join(p, "config.json"))) for n, p in path.items()}
KEYS = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim",
        "intermediate_size", "vocab_size", "max_position_embeddings", "rope_theta", "tie_word_embeddings", "torch_dtype", "transformers_version"]
W = {n: len(n) + 1 for n in conf}                  # 열 너비는 모델 이름 길이 + 1(책 한 줄에 들어가게)
print(f"{'':<23}" + "".join(f"{n:>{W[n]}}" for n in conf))
for k in KEYS:
    print(f"{k:<23}" + "".join(f"{str(c.get(k)):>{W[n]}}" for n, c in conf.items()))

In [ ]:
def count_from_config(c):
    """설정만으로 파라미터 수를 센다: 토큰 표 + 블록마다(어텐션 네 층 + MLP 세 층 + 정규화) + 마지막 정규화(+ 출력층)."""
    d, L, h, kv, hd, ff, V = (c["hidden_size"], c["num_hidden_layers"], c["num_attention_heads"], c["num_key_value_heads"],
                              c.get("head_dim") or c["hidden_size"] // c["num_attention_heads"], c["intermediate_size"], c["vocab_size"])
    attn = d * h * hd + 2 * d * kv * hd + h * hd * d            # 질의, 키·값(GQA), 출력
    mlp = 3 * d * ff                                            # SwiGLU의 게이트·위·아래 층
    norms = 2 * d + (2 * hd if c["model_type"] in ("qwen3", "exaone4") else 0)   # 층 정규화 둘(+ 질의·키 정규화)
    table = V * d
    return table + L * (attn + mlp + norms) + d + (0 if c.get("tie_word_embeddings") else V * d), table

def count_from_files(p):
    """가중치 파일(safetensors)의 머리말만 읽어 텐서마다 원소 수를 더한다. 가중치 값은 읽지 않는다."""
    total = 0
    for f in sorted(os.listdir(p)):
        if f.endswith(".safetensors"):
            with open(os.path.join(p, f), "rb") as fh:
                n = int.from_bytes(fh.read(8), "little")
                head = json.loads(fh.read(n))
            total += sum(math.prod(v["shape"]) for k, v in head.items() if k != "__metadata__")
    return total

def head_same(p):
    """파일에 출력층(lm_head)이 따로 들어 있으면, 그 값이 토큰 표와 같은지 본다. 따로 없으면 None."""
    from safetensors import safe_open
    emb = head = None
    for f in sorted(os.listdir(p)):
        if f.endswith(".safetensors"):
            with safe_open(os.path.join(p, f), "pt") as s:
                for k in s.keys():
                    if k.endswith("embed_tokens.weight"):
                        emb = s.get_tensor(k)
                    if k == "lm_head.weight":
                        head = s.get_tensor(k)
    return None if head is None else bool(torch.equal(emb, head))

api = HfApi()
size = {}
for n, c in conf.items():
    est, table = count_from_config(c)
    real = count_from_files(path[n]) if n in RUN else api.model_info(MODELS[n], files_metadata=False).safetensors.total
    kv_tok = 2 * c["num_hidden_layers"] * c["num_key_value_heads"] * (c.get("head_dim") or c["hidden_size"] // c["num_attention_heads"]) * 2
    size[n] = dict(est=est, real=real, table=table, kv_tok=kv_tok, ctx=c["max_position_embeddings"], same=head_same(path[n]) if n in RUN else None)
    print(f"{n:<22} 설정으로 센 수 {est:>13,}  파일(또는 Hub)의 수 {real:>13,}  {'같음' if est == real else '다름'}  "
          f"토큰 표 {table / est:5.1%}  16비트 가중치 {est * 2 / 2**30:5.2f}GiB  "
          f"토큰 하나의 KV 캐시 {kv_tok / 1024:5.0f}KiB  최대 길이 {c['max_position_embeddings']:,}토큰이면 {kv_tok * c['max_position_embeddings'] / 2**30:5.2f}GiB  "
          f"파일의 출력층이 토큰 표와 같음 {size[n]['same']}")

## 9.2 라이선스 읽기

In [ ]:
# 라이선스 원문에서 상업적 사용과 조건에 관한 문장을 찾아 본다(찾은 문장의 앞부분만 보인다)
PATTERNS = ["commercial", "monthly active users", "Powered by", "research and educational", "at the beginning of", "Apache License"]
for n, p in path.items():
    text = open(os.path.join(p, "LICENSE"), encoding="utf-8").read()
    sents = re.split(r"(?<=[.;:])\s+", " ".join(text.split()))
    print(f"== {n}: {text.split(chr(10))[0].strip()[:60]} ({len(text.split()):,}단어)")
    for pat in PATTERNS:
        hit = [s for s in sents if pat.lower() in s.lower()]
        if hit:
            print(f"   [{pat}] {len(hit)}문장. 예: {hit[0][:110]}")

## 9.3 토크나이저 비교

In [ ]:
tok = {n: AutoTokenizer.from_pretrained(path[n]) for n in RUN}   # 가중치까지 받은 세 모델의 토크나이저
nsmc = [line.split("\t")[1] for line in open("ratings_test.txt", encoding="utf-8").read().split("\n")[1:] if line.count("\t") == 2]
nsmc = [s for s in nsmc if s.strip()][:10000]
chars = sum(len(s) for s in nsmc)

# 2장처럼 NSMC 학습 파일로 어휘 8,000개 바이트 수준 BPE를 직접 학습시켜 함께 견준다
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
train = [line.split("\t")[1] for line in open("ratings_train.txt", encoding="utf-8").read().split("\n")[1:] if line.count("\t") == 2]
own = Tokenizer(models.BPE())
own.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
own.decoder = decoders.ByteLevel()
own.train_from_iterator(train, trainers.BpeTrainer(vocab_size=8000, show_progress=False, initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))

def ntok(name, texts):
    if name == "2장식(NSMC, 8,000)":
        return sum(len(e.ids) for e in own.encode_batch(texts))
    return sum(len(x) for x in tok[name](texts, add_special_tokens=False)["input_ids"])

PAIRS = [("대한민국의 수도는 서울이며, 인구는 약 940만 명이다.", "The capital of South Korea is Seoul, with a population of about 9.4 million."),
         ("이 영화는 배우들의 연기가 좋았지만 이야기가 너무 길었다.", "The actors were good in this movie, but the story was too long."),
         ("물은 섭씨 100도에서 끓고 0도에서 언다.", "Water boils at 100 degrees Celsius and freezes at 0 degrees."),
         ("세종대왕은 1443년에 훈민정음을 만들었다.", "King Sejong created Hunminjeongeum in 1443."),
         ("내일 오전 10시에 회의실에서 만나요.", "Let's meet in the meeting room at 10 a.m. tomorrow.")]
ko, en = [a for a, _ in PAIRS], [b for _, b in PAIRS]
ratio = {}
for name in list(tok) + ["2장식(NSMC, 8,000)"]:
    n = ntok(name, nsmc)
    ratio[name] = chars / n
    vocab = len(tok[name]) if name in tok else own.get_vocab_size()
    print(f"{name:<22} 어휘 {vocab:>7,}  NSMC 1만 문장 {n:>7,}토큰(토큰당 {chars / n:.2f}글자)  "
          f"한국어 5문장 {ntok(name, ko):>3}토큰  같은 뜻의 영어 5문장 {ntok(name, en):>3}토큰")
print(f"NSMC 1만 문장의 글자 수 {chars:,}")
for name in tok:
    print(f"{name:<22}", " | ".join(tok[name].convert_tokens_to_string([t]) for t in tok[name].tokenize("배우들의 연기가 좋았지만")))

## 9.4 채팅 템플릿

In [ ]:
msgs = [{"role": "user", "content": "대한민국의 수도는 어디인가요?"}]
for n in tok:
    extra = {"enable_thinking": False} if n == "Qwen3-0.6B" else {}
    text = tok[n].apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **extra)
    ids = tok[n].apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, **extra)
    print(f"== {n}: {len(ids)}토큰")
    print(text)
print("== Qwen3-0.6B, 생각 모드를 끄지 않은 기본값")
print(tok["Qwen3-0.6B"].apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))

In [ ]:
QA = [("대한민국의 수도는 어디인가요?", ["서울"]), ("일본의 수도는 어디인가요?", ["도쿄"]), ("물의 화학식은 무엇인가요?", ["H2O", "H₂O"]),
      ("1년은 몇 개월인가요?", ["12개월", "12 개월", "12달"]), ("세종대왕이 만든 문자는 무엇인가요?", ["한글", "훈민정음"]), ("태양계에서 가장 큰 행성은 무엇인가요?", ["목성"]),
      ("지구에서 가장 넓은 바다는 무엇인가요?", ["태평양"]), ("7 곱하기 8은 얼마인가요?", ["56"]), ("100에서 37을 빼면 얼마인가요?", ["63"]),
      ("남한에서 가장 높은 산은 무엇인가요?", ["한라산"]), ("임진왜란이 일어난 해는 언제인가요?", ["1592"]), ("삼각형 세 내각의 합은 몇 도인가요?", ["180"]),
      ("사과를 영어로 하면 무엇인가요?", ["apple", "Apple"]), ("서울을 가로질러 흐르는 큰 강은 무엇인가요?", ["한강"]), ("대한민국 국기의 이름은 무엇인가요?", ["태극기"]),
      ("1킬로미터는 몇 미터인가요?", ["1000", "1,000"]), ("식물 세포에서 광합성을 하는 기관은 무엇인가요?", ["엽록체"]),
      ("컴퓨터에서 1바이트는 몇 비트인가요?", ["8비트", "8 비트", "8개"]), ("소설 '어린 왕자'를 쓴 작가는 누구인가요?", ["생텍쥐페리"]), ("빛의 삼원색은 빨강, 초록과 무엇인가요?", ["파랑", "파란"])]
# 정답 낱말이 짧으면 엉뚱한 답도 맞은 것으로 세므로, 숫자는 단위까지 붙여 좁힌다

# 멈출 토큰: 설정 파일(config.json)에 적힌 끝 토큰과, 모델 카드·토크나이저가 안내하는 끝 토큰
CARD_STOP = {"Qwen3-0.6B": ["<|im_end|>", "<|endoftext|>"], "HyperCLOVAX-SEED-0.5B": ["<|endofturn|>", "<|stop|>"],
             "Kanana-1.5-2.1B": ["<|eot_id|>"]}

def run(model, t, prompt_ids, stops, n=64):
    """탐욕 디코딩으로 n토큰까지 만든다. stops 가운데 하나가 나오면 멈춘다."""
    x = torch.tensor([prompt_ids])
    lap = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(x, attention_mask=torch.ones_like(x), max_new_tokens=n, do_sample=False, eos_token_id=stops,
                             temperature=None, top_p=None, top_k=None,       # 모델의 기본 생성 설정에 뽑기가 들어 있어도 끈다
                             pad_token_id=t.pad_token_id if t.pad_token_id is not None else stops[0])
    new = out[0, x.shape[1]:].tolist()
    return new, time.perf_counter() - lap

result, sample, hits = {}, {}, {}
for n in RUN:
    model = AutoModelForCausalLM.from_pretrained(path[n], torch_dtype=torch.float32)
    model.eval()
    shared = model.get_input_embeddings().weight.data_ptr() == model.get_output_embeddings().weight.data_ptr()
    print(f"{n}: 불러온 모델의 파라미터 {sum(p.numel() for p in model.parameters()):,}개(같은 텐서는 한 번만 센다), "
          f"토큰 표와 출력층이 같은 메모리 {shared}, 기본 생성 설정의 do_sample {model.generation_config.do_sample}")
    t = tok[n]
    card = [t.convert_tokens_to_ids(s_) for s_ in CARD_STOP[n]]
    conf_eos = conf[n]["eos_token_id"]
    conf_eos = conf_eos if isinstance(conf_eos, list) else [conf_eos]
    extra = {"enable_thinking": False} if n == "Qwen3-0.6B" else {}
    modes = [("템플릿", card), ("그냥", card)] + ([("템플릿, 설정의 끝 토큰", conf_eos)] if not set(conf_eos) <= set(card) else [])   # 설정의 끝 토큰이 모델 카드와 다를 때만
    print(f"   안내된 끝 토큰 {card}, 설정 파일의 끝 토큰 {conf_eos}")
    for mode, stops in modes:
        right, ended, used, secs = 0, 0, 0, 0.0
        hits[n, mode] = []
        for k, (q, keys) in enumerate(QA):
            ids = (t.apply_chat_template([{"role": "user", "content": q}], tokenize=True, add_generation_prompt=True, **extra)
                   if mode.startswith("템플릿") else t(q)["input_ids"])
            new, sec = run(model, t, ids, stops)
            text = t.decode(new, skip_special_tokens=True)
            right += any(key in text for key in keys); ended += any(i in stops for i in new)
            hits[n, mode].append(any(key in text for key in keys))
            used += len(new); secs += sec
            if k in (0, 7, 18):
                sample[n, mode, k] = text.strip().replace("\n", " ")[:80]
        result[n, mode] = (right / len(QA), ended / len(QA), used / len(QA), secs / used)
        print(f"{n:<22} {mode}: 맞힘 {right}/{len(QA)}  끝 토큰으로 멈춤 {ended}/{len(QA)}  평균 {used / len(QA):.1f}토큰  "
              f"토큰당 {secs / used * 1000:.0f}ms", flush=True)
    del model
for (n, mode, k), text in sample.items():
    print(f"[{n} {mode}] {QA[k][0]} → {text}")

In [ ]:
# Qwen3의 생각 모드: 모델 카드의 권장값(온도 0.6, top-p 0.95, top-k 20)으로 뽑아, 생각에 쓴 토큰과 답을 센다
model = AutoModelForCausalLM.from_pretrained(path["Qwen3-0.6B"], torch_dtype=torch.float32).eval()
t = tok["Qwen3-0.6B"]
END_THINK = t.convert_tokens_to_ids("</think>")
think = []
for k, (q, keys) in enumerate(QA[:10]):
    ids = t.apply_chat_template([{"role": "user", "content": q}], tokenize=True, add_generation_prompt=True)
    torch.manual_seed(k)
    x = torch.tensor([ids])
    lap = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(x, attention_mask=torch.ones_like(x), max_new_tokens=768, do_sample=True, temperature=0.6, top_p=0.95, top_k=20)
    new = out[0, x.shape[1]:].tolist()
    cut = new.index(END_THINK) + 1 if END_THINK in new else len(new)
    answer = t.decode(new[cut:], skip_special_tokens=True)
    think.append((cut, len(new) - cut, any(key in answer for key in keys), time.perf_counter() - lap))
    print(f"{q:<28} 생각 {cut:>4}토큰, 답 {len(new) - cut:>3}토큰, 맞힘 {think[-1][2]}, {think[-1][3]:.1f}초", flush=True)
th = np.array([(a, b, c, d) for a, b, c, d in think], dtype=float)
print(f"10문항: 생각 토큰 중앙값 {np.median(th[:, 0]):.0f}, 맞힘 {int(th[:, 2].sum())}/10, 문항당 {th[:, 3].mean():.1f}초")
print(f"같은 10문항, 생각 모드를 끄고 탐욕 디코딩(앞 셀): 맞힘 {sum(hits['Qwen3-0.6B', '템플릿'][:10])}/10")
del model

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
for fp in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf")) + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(fp)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 9, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"
COLOR = {"Qwen3-0.6B": BLUE, "HyperCLOVAX-SEED-0.5B": GREEN, "EXAONE-4.0-1.2B": PURPLE, "Kanana-1.5-2.1B": ORANGE, "2장식(NSMC, 8,000)": GRAY}
SHORT = {"Qwen3-0.6B": "Qwen3", "HyperCLOVAX-SEED-0.5B": "HyperCLOVA X", "EXAONE-4.0-1.2B": "EXAONE", "Kanana-1.5-2.1B": "Kanana", "2장식(NSMC, 8,000)": "2장식(NSMC, 8,000)"}

fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.4), dpi=200, gridspec_kw={"width_ratios": [1.15, 1]})
ax = axes[0]
names = list(ratio)
ax.barh([SHORT[n] for n in names], [ratio[n] for n in names], color=[COLOR[n] for n in names], height=0.6)
for i, n in enumerate(names):
    ax.text(ratio[n], i, f" {ratio[n]:.2f}", va="center", fontsize=7.6)
ax.invert_yaxis(); ax.set_xlabel("토큰 하나가 담는 글자 수(NSMC 1만 문장)"); ax.tick_params(axis="y", labelsize=7.6)
ax.set_xlim(0, max(ratio.values()) * 1.2)
ax.set_title("① 한국어 영화평", loc="left", fontsize=8.4, color="#374151")
ax = axes[1]
x = np.arange(len(tok))
ax.bar(x - 0.2, [ntok(n, ko) for n in tok], 0.38, color=[COLOR[n] for n in tok])
ax.bar(x + 0.2, [ntok(n, en) for n in tok], 0.38, color="white", edgecolor=[COLOR[n] for n in tok], hatch="///")
ax.bar([0], [0], color=GRAY, label="한국어(색 막대)"); ax.bar([0], [0], color="white", edgecolor=GRAY, hatch="///", label="같은 뜻의 영어(빗금)")
ax.set_xticks(x); ax.set_xticklabels([SHORT[n] for n in tok], fontsize=7.6)
ax.set_ylabel("다섯 문장의 토큰 수"); ax.legend(frameon=False, fontsize=7.2)
ax.set_title("② 같은 뜻, 다른 언어", loc="left", fontsize=8.4, color="#374151")
for a in axes:
    a.grid(axis="x" if a is axes[0] else "y", color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_9_tokens.png"); plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.2), dpi=200)
names = list(size)
ax = axes[0]
est = [size[n]["est"] / 1e9 for n in names]; real = [size[n]["real"] / 1e9 for n in names]; table = [size[n]["table"] / 1e9 for n in names]
short = ["Qwen3", "HyperCLOVA X", "EXAONE", "Kanana"]
ax.bar(short, table, color=GRAY, label="토큰 표(출력층과 공유)")
ax.bar(short, [e - t_ for e, t_ in zip(est, table)], bottom=table, color=[COLOR[n] for n in names], label="블록")
ax.bar(short, [r - e for r, e in zip(real, est)], bottom=est, color="white", edgecolor=GRAY, hatch="///", label="파일에만 있는 출력층 사본")
for i in range(len(names)):
    ax.text(i, real[i] + 0.03, f"{est[i]:.2f}" + (f"\n(파일 {real[i]:.2f})" if real[i] != est[i] else ""), ha="center", fontsize=6.8)
ax.set_ylim(0, max(real) * 1.25)
ax.set_ylabel("파라미터(10억 개)"); ax.legend(frameon=False, fontsize=6.8, loc="upper left"); ax.tick_params(axis="x", labelsize=6.8)
ax.set_title("① 파라미터와 토큰 표의 몫", loc="left", fontsize=8.4, color="#374151")
ax = axes[1]
for n in names:
    L = np.array([1024, 4096, 8192, 32768])
    L = L[L <= size[n]["ctx"]]
    ax.plot(L, size[n]["kv_tok"] * L / 2**30, "-o", ms=3.5, color=COLOR[n], label=SHORT[n])
ax.axhline(size["Kanana-1.5-2.1B"]["real"] * 2 / 2**30, color=ORANGE, ls=":", lw=1)
ax.text(1100, size["Kanana-1.5-2.1B"]["real"] * 2 / 2**30 * 0.93, "Kanana 가중치 파일(16비트)", fontsize=6.8, color=ORANGE)
ax.set_xscale("log", base=2); ax.set_xticks([1024, 4096, 8192, 32768]); ax.set_xticklabels(["1,024", "4,096", "8,192", "32,768"], fontsize=7)
ax.xaxis.set_minor_formatter(plt.NullFormatter()); ax.set_xlabel("문장 길이(토큰, 로그 눈금)"); ax.set_ylabel("문장 하나의 KV 캐시(GiB, 16비트)")
ax.legend(frameon=False, fontsize=6.8)
ax.set_title("② 길이에 따른 KV 캐시", loc="left", fontsize=8.4, color="#374151")
for a in axes:
    a.grid(axis="y", color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_9_size.png"); plt.close(fig)